**Parameters**

Tagged as a Fabric parameter cell (cell "..." menu → "Toggle parameter
cell"). A calling Data Factory pipeline overrides these at runtime.

In [1]:
# PARAMETERS CELL — tag this cell as a parameter cell in Fabric
# (cell "..." menu -> Toggle parameter cell)

processing_date = "2026-08-28"   # overridden by pipeline; used for logging/traceability
watermark_table_name = "customers"  # which source's watermark this run processes

StatementMeta(, 4595c577-0d8a-46f6-86f8-1e9339a2bc73, 3, Finished, Available, Finished, False)

**CDC-Style Incremental Load (Timestamp Watermark)**

Tracks the last successful load timestamp per table in `pipeline_watermarks`,
then processes only rows changed since that watermark — avoiding full
reprocessing of unchanged data on every run. See ADR-007 for why this
pattern was chosen over log-based CDC.

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, IntegerType
from delta.tables import DeltaTable
from datetime import datetime, timezone

from datetime import datetime, timezone, date as date_type

CDC_RUN_TS = datetime.now(timezone.utc).isoformat()
TODAY = date_type.fromisoformat(processing_date)  # now driven by the parameter, not hardcoded
print(f"CDC layer run started at {CDC_RUN_TS}, processing_date={TODAY}")

# Create the watermark table if it doesn't exist yet
if not spark.catalog.tableExists("pipeline_watermarks"):
    empty_watermarks = spark.createDataFrame(
        [], "table_name STRING, last_watermark_ts TIMESTAMP"
    )
    empty_watermarks.write.format("delta").saveAsTable("pipeline_watermarks")
    print("Created empty pipeline_watermarks table.")


def get_watermark(table_name: str):
    """Returns the last watermark timestamp for a table (as a raw timestamp value),
    or a far-past default string if no watermark has been recorded yet."""
    result = spark.table("pipeline_watermarks").filter(F.col("table_name") == table_name)
    if result.count() == 0:
        return "1900-01-01 00:00:00"
    return result.collect()[0]["last_watermark_ts"]


def set_watermark(table_name: str, new_ts):
    """Upserts the watermark for a table. new_ts must be a datetime/timestamp value,
    not a string — Delta's declared TimestampType column will reject a plain string."""
    wm_table = DeltaTable.forName(spark, "pipeline_watermarks")
    new_row = spark.createDataFrame([(table_name, new_ts)], "table_name STRING, last_watermark_ts TIMESTAMP")
    wm_table.alias("t").merge(
        new_row.alias("s"), "t.table_name = s.table_name"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


print(f"CDC run started at {CDC_RUN_TS}")

StatementMeta(, 4595c577-0d8a-46f6-86f8-1e9339a2bc73, 4, Finished, Available, Finished, False)

CDC layer run started at 2026-08-28T17:54:57.882811+00:00, processing_date=2026-08-28
CDC run started at 2026-08-28T17:54:57.882811+00:00


**Incremental Load: Customers**

Reads only `bronze_customers` rows where `updated_at` is newer than the
last recorded watermark, applies the same cleaning logic as the full
Silver build (Phase 6), then MERGEs (upserts) into `silver_customers`
rather than overwriting the whole table. On the very first run, the
watermark defaults to 1900-01-01, so every row is "new" — this establishes
the baseline. Subsequent runs will only touch genuinely changed rows.

In [3]:
last_wm = get_watermark(watermark_table_name)
print(f"Last watermark for {watermark_table_name}: {last_wm}")

bronze_customers = spark.table("bronze_customers")

changed_customers = bronze_customers.filter(
    F.to_timestamp(F.col("updated_at")) > F.lit(last_wm)
)

changed_count = changed_customers.count()
print(f"Rows changed since last watermark: {changed_count}")

if changed_count > 0:
    changed_clean = (
        changed_customers
        .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
        .withColumn(
            "email",
            F.when((F.trim(F.col("email")) == "") | F.col("email").isNull(), None)
             .otherwise(F.lower(F.trim(F.col("email"))))
        )
        .withColumn("phone", F.regexp_replace(F.col("phone"), r"[\s\-\(\)]", ""))
        .select("customer_id", "first_name", "last_name", "email", "phone", "city", "country")
        .dropDuplicates(["customer_id"])
    )

    if not spark.catalog.tableExists("silver_customers_incremental_demo"):
        changed_clean.write.format("delta").saveAsTable("silver_customers_incremental_demo")
        print(f"Created silver_customers_incremental_demo with {changed_clean.count()} rows.")
    else:
        target = DeltaTable.forName(spark, "silver_customers_incremental_demo")
        target.alias("t").merge(
            changed_clean.alias("s"), "t.customer_id = s.customer_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        print(f"Merged {changed_count} changed rows into silver_customers_incremental_demo.")

    max_updated = bronze_customers.agg(F.max(F.to_timestamp("updated_at"))).collect()[0][0]
    set_watermark(watermark_table_name, max_updated)
    print(f"Watermark updated to: {max_updated}")
else:
    print("No changes since last watermark — nothing to process.")

StatementMeta(, 4595c577-0d8a-46f6-86f8-1e9339a2bc73, 5, Finished, Available, Finished, False)

Last watermark for customers: 2026-08-28 08:49:32.938586
Rows changed since last watermark: 0
No changes since last watermark — nothing to process.


In [4]:
spark.table("silver_customers_incremental_demo").filter(F.col("customer_id").isin([100, 200])).select(
    "customer_id", "city"
).show()

print(f"Total rows in silver_customers_incremental_demo: {spark.table('silver_customers_incremental_demo').count()}")

StatementMeta(, 4595c577-0d8a-46f6-86f8-1e9339a2bc73, 6, Finished, Available, Finished, False)

+-----------+-------+
|customer_id|   city|
+-----------+-------+
|        100|Nairobi|
|        200|Nairobi|
+-----------+-------+

Total rows in silver_customers_incremental_demo: 520
